In [ ]:
!apt update
!apt install nvidia-cuda-toolkit -y

In [9]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [6]:
%%writefile matrixMul.cu

#include <iostream>
#include <cuda_runtime.h>

using namespace std;

__global__ void matrixMul(int *A, int *B, int *C, int rowsA, int colsA, int colsB) {

    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if(row < rowsA && col < colsB) {

        int sum = 0;

        for(int k = 0; k < colsA; k++) {
            sum += A[row * colsA + k] * B[k * colsB + col];
        }

        C[row * colsB + col] = sum;
    }
}

int main() {

    int rowsA = 2, colsA = 2;
    int rowsB = 2, colsB = 2;

    int h_A[] = {1, 2,
                 3, 4};

    int h_B[] = {5, 6,
                 7, 8};

    int h_C[4];

    int *d_A, *d_B, *d_C;

    int sizeA = rowsA * colsA * sizeof(int);
    int sizeB = rowsB * colsB * sizeof(int);
    int sizeC = rowsA * colsB * sizeof(int);

    cudaMalloc((void**)&d_A, sizeA);
    cudaMalloc((void**)&d_B, sizeB);
    cudaMalloc((void**)&d_C, sizeC);

    cudaMemcpy(d_A, h_A, sizeA, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, sizeB, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(2,2);
    dim3 blocksPerGrid(1,1);

    matrixMul<<<blocksPerGrid, threadsPerBlock>>>(d_A,d_B,d_C,rowsA,colsA,colsB);

    cudaMemcpy(h_C, d_C, sizeC, cudaMemcpyDeviceToHost);

    cout << "Result Matrix:\n";

    for(int i=0;i<rowsA;i++) {
        for(int j=0;j<colsB;j++) {
            cout << h_C[i * colsB + j] << " ";
        }
        cout << endl;
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Overwriting matrixMul.cu


In [7]:
!nvcc matrixMul.cu -o matrixMul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [8]:
!./matrixMul

Result Matrix:
19 22 
43 50 
